# 02 · Backtesting: baselines vs. a global LightGBM model, with prediction intervals
**Rolling-origin backtest:** 3 folds, each forecasting the 48 days after its cutoff (the same horizon as the Kaggle test).
Store statistics are fit only on data before each cutoff. The number of boosting rounds is chosen *once* on an inner
validation window that ends before every evaluation window, so no evaluation data is used for tuning. **Runtime: CPU**, about 20–30 min.

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FC_SMOKE") == "1"
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["FC_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/forecast/src/forecast/models.py", recursive=True), key=len)
    assert hits, "Upload the 'forecast' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate", "fastapi", "uvicorn", "httpx"], check=True)
else:
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src"); sys.path.insert(0, PROJECT_DIR)
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data/rossmann" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from forecast import data as D, models as M
from forecast.features import FEATURES, build_features, fit_store_stats
if SMOKE:
    train, test, store = D.make_synthetic(25)
else:
    train, test, store = D.load(D.download(DATA))
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=str)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"train {train.shape} {train.Date.min().date()}..{train.Date.max().date()} | test {test.shape} | stores {store.shape[0]}")

In [ ]:
folds = M.make_folds(train, n_folds=3, horizon=48)
for f in folds: print(f["fold"], "train <=", f["cutoff"].date(), "| forecast", (f["cutoff"] + pd.Timedelta(days=1)).date(), "->", f["valid_end"].date())
# pick the number of boosting rounds on an inner window before the earliest evaluation fold
earliest = folds[-1]["cutoff"]
inner_hist = train[train.Date <= earliest - pd.Timedelta(days=48)]
inner_val = train[(train.Date > earliest - pd.Timedelta(days=48)) & (train.Date <= earliest)]
t0 = time.time()
m_inner = M.fit_lgbm(inner_hist, store, num_boost_round=300 if SMOKE else 4000, valid=inner_val)
N_ROUNDS = max(100, int(m_inner.booster.best_iteration * 1.1))
print(f"rounds = {N_ROUNDS}  ({time.time()-t0:.0f}s)")

## 1. Backtest

In [ ]:
rows, fold_preds = [], {}
for f in folds:
    hist, fut = M.split(train, f)
    t0 = time.time()
    model = M.fit_lgbm(hist, store, num_boost_round=N_ROUNDS)
    preds = {"Seasonal naive (same weekday, last 6 wks)": M.baseline_last_weeks(hist, fut),
             "Store x weekday x promo median": M.baseline_store_dow_promo(hist, fut),
             "LightGBM (global)": model.predict(fut, store)}
    fold_preds[f["fold"]] = (fut, preds)
    for name, p in preds.items():
        rows.append({"fold": f["fold"], "model": name, **M.metrics(fut.Sales, p)})
    print(f"fold {f['fold']}: LightGBM RMSPE {M.rmspe(fut.Sales, preds['LightGBM (global)']):.4f}  ({time.time()-t0:.0f}s)")
bt = pd.DataFrame(rows)
summary = bt.groupby("model")[["rmspe", "wape", "mae"]].agg(["mean", "std"]).sort_values(("rmspe", "mean"))
bt.to_csv(f"{ART}/backtest_folds.csv", index=False); summary.round(4)

## 2. Bias correction
RMSPE punishes over-forecasts more than under-forecasts, and back-transforming from log sales adds bias. A single multiplicative
factor is fit on the *earlier* folds and checked on the latest fold.

In [ ]:
def best_factor(y, p):
    grid = np.linspace(0.94, 1.04, 51)
    return float(grid[np.argmin([M.rmspe(y, p * g) for g in grid])])
fut1, pr1 = fold_preds[1]; fut2, pr2 = fold_preds[2]
FACTOR = best_factor(np.r_[fut1.Sales.values, fut2.Sales.values], np.r_[pr1["LightGBM (global)"], pr2["LightGBM (global)"]])
fut0, pr0 = fold_preds[0]
print(f"factor {FACTOR:.3f} (fit on folds 1-2) | fold 0 RMSPE: {M.rmspe(fut0.Sales, pr0['LightGBM (global)']):.4f} -> "
      f"{M.rmspe(fut0.Sales, pr0['LightGBM (global)'] * FACTOR):.4f}")

## 3. Where are the errors? By forecast horizon, weekday and store type (latest fold)

In [ ]:
e = fut0[["Store", "Date", "DayOfWeek", "Sales", "Open"]].copy(); e["pred"] = pr0["LightGBM (global)"] * FACTOR
e = e[(e.Open == 1) & (e.Sales > 0)].merge(store[["Store", "StoreType"]])
e["ape"] = (e.Sales - e.pred).abs() / e.Sales; e["horizon"] = (e.Date - folds[0]["cutoff"]).dt.days
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
e.groupby("horizon").ape.mean().plot(ax=ax[0]); ax[0].set_title("Mean abs. % error vs days ahead"); ax[0].set_xlabel("days ahead")
e.groupby("DayOfWeek").ape.mean().plot.bar(ax=ax[1], color="#4C72B0"); ax[1].set_title("by weekday")
e.groupby("StoreType").ape.mean().plot.bar(ax=ax[2], color="#55A868"); ax[2].set_title("by store type")
savefig("02_error_breakdown")
worst = e.groupby("Store").ape.mean().sort_values(ascending=False).head(5); print("worst stores (MAPE):", worst.round(3).to_dict())

## 4. Feature-group ablation (latest fold): what does each group contribute?

In [ ]:
groups = {
    "store history stats": [f for f in FEATURES if f.startswith("store_")],
    "neighbouring-day context": ["holiday_tomorrow", "holiday_yesterday", "closed_tomorrow", "closed_yesterday", "promo_day_index", "promo_tomorrow"],
    "Promo2 & competition": ["comp_months_open", "log_comp_distance", "CompetitionDistance", "Promo2", "promo2_weeks", "promo2_active_month"],
    "calendar": ["year", "month", "day", "weekofyear", "dayofyear", "days_to_month_end", "is_december", "days_to_christmas"]}
hist0, fut0_ = M.split(train, folds[0]); full_rmspe = M.rmspe(fut0_.Sales, pr0["LightGBM (global)"])
abl = [{"removed": "(none: full model)", "rmspe": full_rmspe, "delta": 0.0}]
for g, cols in groups.items():
    feats = [f for f in FEATURES if f not in cols]
    p = M.fit_lgbm(hist0, store, features=feats, num_boost_round=N_ROUNDS).predict(fut0_, store)
    abl.append({"removed": g, "rmspe": M.rmspe(fut0_.Sales, p), "delta": M.rmspe(fut0_.Sales, p) - full_rmspe}); print(abl[-1])
abl = pd.DataFrame(abl); abl.to_csv(f"{ART}/ablation.csv", index=False); abl.round(4)

## 5. Prediction intervals: split conformal, with coverage checked on unseen data
Calibrate on fold 1 (residuals on the log scale), then measure how often fold 0's actual sales fall inside the intervals.

In [ ]:
Q = M.conformal_quantiles(fut1.Sales.values, pr1["LightGBM (global)"] * FACTOR, levels=(0.8, 0.95))
cov_rows = []
y0, p0 = fut0.Sales.values, pr0["LightGBM (global)"] * FACTOR
st0 = fut0[["Store"]].merge(store[["Store", "StoreType"]]).StoreType.values
for lv, q in Q.items():
    lo, hi = M.interval(p0, q)
    row = {"nominal": lv, "log_halfwidth": q, "coverage_overall": M.coverage(y0, lo, hi),
           "median_width_%": float(np.median(((hi - lo) / np.maximum(p0, 1))[y0 > 0]) * 100)}
    for t in sorted(set(st0)):
        row[f"coverage_type_{t}"] = M.coverage(y0, lo, hi, mask=(y0 > 0) & (st0 == t))
    cov_rows.append(row)
cov = pd.DataFrame(cov_rows); cov.round(3)

In [ ]:
s = fut0.assign(pred=p0)[lambda d: d.Store == d.Store.iloc[0]].sort_values("Date")
lo80, hi80 = M.interval(s.pred.values, Q[0.8]); lo95, hi95 = M.interval(s.pred.values, Q[0.95])
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.fill_between(s.Date, lo95, hi95, alpha=.15, label="95% interval"); ax.fill_between(s.Date, lo80, hi80, alpha=.3, label="80% interval")
ax.plot(s.Date, s.pred, label="forecast"); ax.plot(s.Date, s.Sales, "k.", label="actual")
ax.set_title(f"Store {s.Store.iloc[0]}: 48-day forecast with conformal intervals (unseen fold)"); ax.legend(fontsize=8)
savefig("02_forecast_example")

In [ ]:
imp = pd.Series(model.booster.feature_importance("gain"), index=model.features).sort_values().tail(15)
imp.plot.barh(figsize=(6, 5), color="#4C72B0", title="LightGBM feature importance (gain)"); savefig("02_importance")
save_json({"n_rounds": N_ROUNDS, "factor": FACTOR, "summary": {m: {k: float(v) for k, v in g.items()} for m, g in bt.groupby("model")[["rmspe", "wape"]].mean().iterrows()},
           "per_fold": bt.to_dict(orient="records"), "conformal_q": Q, "coverage": cov.to_dict(orient="records")}, "backtest")
print("saved")

## Observations (fill in yourself)
* How much better than the baselines is the global model, and is it stable across folds?
* Which feature groups matter most? Do the intervals achieve their nominal coverage, and for which store types not?